# Snowflake Classes — Complete Notes

---

## Is it the same as OOP (Object-Oriented Programming)?

**No. Snowflake "classes" are NOT the same as classes in Python/Java/C++.**

However, Snowflake **borrows the terminology** from OOP. Here is the mapping:

| OOP Concept | Snowflake Equivalent | Meaning |
|---|---|---|
| Class (blueprint) | **Class** (e.g. `ANOMALY_DETECTION`) | A built-in schema-level object type defined by Snowflake. You cannot create your own classes. |
| Object / Instance | **Instance** | A concrete object you create FROM a class using `CREATE ... <instance_name>`. It holds trained state or configuration. |
| Method | **Method** | A function you call ON an instance (e.g. `instance_name!DETECT_ANOMALIES(...)`) |
| Constructor | **CREATE statement** | The DDL that builds an instance from training data or config |

### Key Differences from OOP

1. **You cannot define your own classes** — Snowflake provides a fixed set of built-in classes (listed below).
2. **Instances are persistent database objects** — they live in a schema, have owners, and survive sessions (unlike OOP objects which live in memory).
3. **Methods are called with `!` syntax** — e.g. `my_model!FORECAST(...)` not `my_model.forecast()`.
4. **Instances are trained/configured at creation time** — you provide training data in the CREATE statement; the instance stores the learned model internally.

---

## Core Syntax Pattern

```sql
-- Step 1: Create an instance (like instantiating an object)
CREATE <class_name> <instance_name>(...)
  USING (SELECT ... FROM training_data);

-- Step 2: Call a method on the instance
SELECT * FROM TABLE(
  <instance_name>!<METHOD_NAME>(INPUT => ...)
);

-- Step 3: Drop when no longer needed
DROP <class_name> <instance_name>;
```

---

## The 8 Built-in Classes

---

### 1. ANOMALY_DETECTION

**Purpose:** Detect outliers/anomalies in time-series data.

**How it works:** You train it on historical "normal" data. Then call its method on new data to flag anomalies.

```sql
-- Create (train) the model
CREATE SNOWFLAKE.ML.ANOMALY_DETECTION my_anomaly_model(
  INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'training_data'),
  TIMESTAMP_COLNAME => 'ts',
  TARGET_COLNAME => 'value',
  LABEL_COLNAME => ''  -- unsupervised
);

-- Detect anomalies on new data
CALL my_anomaly_model!DETECT_ANOMALIES(
  INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'new_data'),
  TIMESTAMP_COLNAME => 'ts',
  TARGET_COLNAME => 'value'
);
```

**Key method:** `DETECT_ANOMALIES` — returns rows with `IS_ANOMALY` (boolean) and anomaly score.

---

### 2. ANOMALY_INSIGHTS

**Purpose:** Same as ANOMALY_DETECTION but also provides **explanations** for why each point is anomalous.

**Difference from ANOMALY_DETECTION:**
- Returns additional columns explaining feature contributions to each anomaly.
- Helps answer: "Why is this data point flagged?"

---

### 3. FORECAST

**Purpose:** Predict future values of a time-series.

```sql
-- Train the forecasting model
CREATE SNOWFLAKE.ML.FORECAST my_forecast_model(
  INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'historical_sales'),
  TIMESTAMP_COLNAME => 'date',
  TARGET_COLNAME => 'revenue'
);

-- Generate predictions for next 30 days
CALL my_forecast_model!FORECAST(
  FORECASTING_PERIODS => 30
);
```

**Key method:** `FORECAST` — returns predicted values with confidence intervals (upper/lower bounds).

---

### 4. TOP_INSIGHTS

**Purpose:** Automatically find the most significant drivers/segments that explain a metric change.

**Use case:** "Why did revenue drop last week?" — it finds which dimensions (region, product, segment) contributed most.

```sql
-- Create instance
CREATE SNOWFLAKE.ML.TOP_INSIGHTS my_insights(
  INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'metric_data'),
  METRIC_COLNAME => 'revenue',
  DIMENSION_COLNAMES => ['region', 'product', 'channel']
);

-- Get insights
CALL my_insights!GET_INSIGHTS();
```

**Key method:** `GET_INSIGHTS` — returns ranked list of segments that explain the metric movement.

---

### 5. BUDGET

**Purpose:** Monitor and control Snowflake credit consumption.

**This is NOT an ML model.** It is a governance/cost-management class.

```sql
-- Create a budget
CREATE SNOWFLAKE.CORE.BUDGET my_budget();

-- Set a spending limit
CALL my_budget!SET_SPENDING_LIMIT(500);  -- 500 credits

-- Check current usage
CALL my_budget!GET_SPENDING_HISTORY();
```

**Key methods:**
- `SET_SPENDING_LIMIT` — set credit cap
- `GET_SPENDING_HISTORY` — view consumption
- `ADD_RESOURCE` / `REMOVE_RESOURCE` — attach/detach warehouses or other resources to the budget

---

### 6. CLASSIFICATION

**Purpose:** Automatically classify columns in a table by their **data type sensitivity** (e.g., PII: name, email, phone, SSN).

**This is a data governance class**, not a predictive ML model.

```sql
-- Classify columns of a table
SELECT * FROM TABLE(
  SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION!CLASSIFY(
    INPUT => SYSTEM$REFERENCE('TABLE', 'customers')
  )
);
```

**Key method:** `CLASSIFY` — returns each column's inferred semantic category and recommended privacy tag.

---

### 7. CLASSIFICATION_PROFILE

**Purpose:** View and manage the profile/results of a prior classification run.

**Relation to CLASSIFICATION:** After `CLASSIFICATION!CLASSIFY` runs, results can be stored as a profile for auditing and tracking changes over time.

---

### 8. CUSTOM_CLASSIFIER

**Purpose:** Create your own classification rules beyond the built-in categories.

**Use case:** You have domain-specific sensitive data (e.g., internal project codes, proprietary IDs) that Snowflake's default classifier doesn't recognize.

```sql
-- Create a custom classifier
CREATE SNOWFLAKE.DATA_PRIVACY.CUSTOM_CLASSIFIER my_classifier();

-- Add a regex-based rule
CALL my_classifier!ADD_REGEX(
  'INTERNAL_PROJECT_CODE',
  'IDENTIFIER',
  'PRJ-[A-Z]{3}-[0-9]{4}'
);

-- Use it to classify
SELECT * FROM TABLE(
  my_classifier!CLASSIFY(
    INPUT => SYSTEM$REFERENCE('TABLE', 'projects')
  )
);
```

**Key methods:** `ADD_REGEX`, `LIST_CATEGORIES`, `CLASSIFY`

---

## Summary Table

| Class | Domain | What the instance stores |
|---|---|---|
| ANOMALY_DETECTION | ML / Time-series | Trained model of "normal" patterns |
| ANOMALY_INSIGHTS | ML / Time-series | Trained model + explanation logic |
| FORECAST | ML / Time-series | Trained forecasting model |
| TOP_INSIGHTS | ML / Analytics | Metric analysis configuration |
| BUDGET | Governance / Cost | Spending limits and resource mappings |
| CLASSIFICATION | Governance / Privacy | Built-in classification engine (singleton) |
| CLASSIFICATION_PROFILE | Governance / Privacy | Stored classification results |
| CUSTOM_CLASSIFIER | Governance / Privacy | User-defined classification rules |

---

## Mental Model

```
┌────────────────────────────────────────────────────────┐
│  OOP World                  Snowflake World            │
│                                                        │
│  class Dog:          ←→     SNOWFLAKE.ML.FORECAST      │
│      (blueprint)            (built-in blueprint)       │
│                                                        │
│  my_dog = Dog()      ←→     CREATE FORECAST my_model   │
│      (instance)             (persistent DB object)     │
│                                                        │
│  my_dog.bark()       ←→     my_model!FORECAST(...)     │
│      (method call)          (method call with !)       │
│                                                        │
│  del my_dog          ←→     DROP FORECAST my_model     │
│      (destroy)              (remove from schema)       │
└────────────────────────────────────────────────────────┘
```

---

## Key Takeaways

1. **Same concept, different context** — Snowflake uses the class/instance/method metaphor but applies it to persistent database objects, not in-memory programming constructs.
2. **You create instances, not classes** — Snowflake defines the classes; you create instances from them.
3. **Instances live in schemas** — they are first-class database objects with ownership, grants, and lifecycle.
4. **The `!` operator** calls methods on instances — this is unique to Snowflake SQL.
5. **Two categories exist:** ML classes (FORECAST, ANOMALY_*, TOP_INSIGHTS) and Governance classes (BUDGET, CLASSIFICATION, CUSTOM_CLASSIFIER).

## Where Do Classes Live?

Classes are **schema-level objects** in the `SNOWFLAKE` database. You must use the fully qualified class name to execute SQL commands.

---

### List Available Classes

```sql
-- All classes in the SNOWFLAKE database
SHOW CLASSES IN DATABASE SNOWFLAKE;

-- Only ML classes
SHOW CLASSES IN SCHEMA SNOWFLAKE.ML;
```

---

### List Methods (Functions) in a Class

```sql
SHOW FUNCTIONS IN CLASS SNOWFLAKE.ML.ANOMALY_DETECTION;
SHOW FUNCTIONS IN CLASS SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE;
```

---

### List Procedures in a Class

```sql
SHOW PROCEDURES IN CLASS SNOWFLAKE.ML.ANOMALY_DETECTION;
SHOW PROCEDURES IN CLASS SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE;
```

---

### List Roles in a Class

```sql
SHOW ROLES IN CLASS SNOWFLAKE.ML.ANOMALY_DETECTION;
SHOW ROLES IN CLASS SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE;
```